# Functions, Tools & Agents

En este notebook veremos cómo crear funciones, herramientas y agentes


## Configuración de la API de OpenAI
Este código importa las librerías necesarias para trabajar con la API de OpenAI, carga las variables de entorno desde un archivo .env y configura la clave de la API de OpenAI para ser utilizada en las peticiones a la API.


In [1]:
!pip install langchain openai langchain_experimental langchain_openai tiktoken wikipedia langchainhub

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 209.2/209.2 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.7 MB/s eta 0:00:00
  Created wheel for wikipedia: filename=wikipedia-1.4.0-py3-none-any.whl size=11678 sha256=cd920d6c36e3de41ce0a35e442f6474d7cf081e283fe26ccd2ba2fe9ba0d9249
  Stored in directory: /root/.cache/pip/wheels/8f/ab/cb/45ccc40522d3a1c41e1d2ad53b8f33a62f394011ec38cd71c6
Successfully built wikipedia


In [2]:
import os
import getpass

api_key = getpass.getpass("Enter your OpenAI API Key:")

Enter your OpenAI API Key:··········


## Function Calling

https://dev.pokemontcg.io/dashboard

In [4]:
import requests

def get_most_expensive_pokemon_card(card_name):
    BASE_URL = "https://api.pokemontcg.io/v2/cards"
    params = {"q": f"name:{card_name}"}
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        cards = response.json().get("data", [])
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    if not cards:
        return "Pokémon card not found"

    most_expensive_card = max(cards, key=lambda card: card.get("tcgplayer", {}).get("prices", {}).get("holofoil", {}).get("market", 0) or 0)
    highest_price = most_expensive_card.get("tcgplayer", {}).get("prices", {}).get("holofoil", {}).get("market", "N/A")

    return f"Most Expensive {card_name} Card: {most_expensive_card['name']} - Market Price: ${highest_price}"

examples = ["Charizard", "Pikachu", "Blastoise", "Venusaur", "Mewtwo"]
for card in examples:
    print(get_most_expensive_pokemon_card(card))


Most Expensive Charizard Card: Charizard ★ δ - Market Price: $996.5
Most Expensive Pikachu Card: Pikachu ★ - Market Price: $2066.41
Most Expensive Blastoise Card: Blastoise - Market Price: $400.0
Most Expensive Venusaur Card: Venusaur ex - Market Price: $283.99
Most Expensive Mewtwo Card: Rocket's Mewtwo ex - Market Price: $1999.99


Definimos nuestra API con la estructura OpenAPI


In [5]:
functions = [
    {
      "name": "get_most_expensive_pokemon_card",
      "description": "Find the most expensive Pokémon card based on the Pokémon's name.",
      "parameters": {
        "type": "object",
        "properties": {
          "pokemon_name": {
            "type": "string",
            "description": "The name of the Pokémon to search for. Example values: 'Charizard', 'Pikachu', 'Blastoise', 'Venusaur', 'Mewtwo'."
          }
        },
        "required": ["pokemon_name"]
      }
    }
]

Definimos la función de completado con funciones como parámetro de entrada


In [6]:
from openai import OpenAI

client = OpenAI(api_key = api_key)

def get_completion_from_functions(messages, model="gpt-4.1-mini", temperature=0, functions = None):
    response = client.chat.completions.create(
        model=model,
        messages=messages,
        temperature=temperature,
        functions=functions
    )
    return response

Ahora pediremos al modelo cuál es el piso más caro de Encamp, a ver si es capaz de decidir utilizar la función que hemos creado.


In [15]:
messages = [
    {
        "role": "user",
        "content": "Cual es la carta más cara de Pikachu?"
    }
]

In [16]:
response = get_completion_from_functions(messages=messages, functions=functions)

In [17]:
print(response)

ChatCompletion(id='chatcmpl-BtJsi6xdm6349FrdXNqYHhwJbObwD', choices=[Choice(finish_reason='function_call', index=0, logprobs=None, message=ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=FunctionCall(arguments='{"pokemon_name":"Pikachu"}', name='get_most_expensive_pokemon_card'), tool_calls=None))], created=1752524164, model='gpt-4.1-mini-2025-04-14', object='chat.completion', service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=23, prompt_tokens=105, total_tokens=128, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0)))


In [18]:
response_message = response.choices[0].message
print(response_message.content)
print(response_message.function_call)
print(response_message.function_call.arguments)

None
FunctionCall(arguments='{"pokemon_name":"Pikachu"}', name='get_most_expensive_pokemon_card')
{"pokemon_name":"Pikachu"}


In [19]:
import json

args = json.loads(response_message.function_call.arguments)
print(args)

{'pokemon_name': 'Pikachu'}


Llamamos a la función utilizando los args que ha sido capaz de extraer de la frase del usuario


In [20]:
get_most_expensive_pokemon_card(args["pokemon_name"])

'Most Expensive Pikachu Card: Pikachu ★ - Market Price: $2066.41'

Ahora intentaremos con una pregunta que no tiene nada que ver, a ver si infiere no tener que utilizar la función


In [21]:
messages = [
    {
        "role": "user",
        "content": "Quien es el tio mas guapo de Andorra?"
    }
]

In [22]:
response = get_completion_from_functions(messages=messages, functions=functions)

In [23]:
response_message = response.choices[0].message
print(response_message.content)
print(response_message.function_call)

No tengo información específica sobre quién es considerado el "tío más guapo de Andorra". La belleza es subjetiva y puede variar según las opiniones de cada persona. ¿Quieres que te ayude con alguna otra consulta?
None


## Tools

Ahora utilizaremos Langchain para definir alguna herramienta para que posteriormente un LLM Agent pueda decidir usar la herramienta.


In [24]:
from typing import Optional
from langchain.agents import tool
from langchain.pydantic_v1 import BaseModel, Field

/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py:3553: LangChainDeprecationWarning: As of langchain-core 0.3.0, LangChain uses pydantic v2 internally. The langchain.pydantic_v1 module was a compatibility shim for pydantic v1, and should no longer be used. Please update the code to import from Pydantic directly.

For example, replace imports like: `from langchain.pydantic_v1 import BaseModel`
with: `from pydantic import BaseModel`
or the v1 compatibility namespace if you are working in a code base that has not been fully upgraded to pydantic 2 yet. 	from pydantic.v1 import BaseModel

  exec(code_obj, self.user_global_ns, self.user_ns)


In [25]:
from typing import Optional
from pydantic import BaseModel, Field
import requests

class PokemonSearchInput(BaseModel):
    pokemon_name: str = Field(description="The name of the Pokémon to search for. Example values: 'Charizard', 'Pikachu', 'Blastoise', 'Venusaur', 'Mewtwo'.")

@tool('get-most-expensive-pokemon-card', args_schema=PokemonSearchInput, return_direct=True)
def get_most_expensive_pokemon_card(pokemon_name: str) -> str:
    """Find the most expensive Pokémon card based on the Pokémon's name."""

    BASE_URL = "https://api.pokemontcg.io/v2/cards"
    params = {"q": f"name:{pokemon_name}"}
    response = requests.get(BASE_URL, params=params)

    if response.status_code == 200:
        cards = response.json().get("data", [])
    else:
        raise Exception(f"API Request failed with status code: {response.status_code}")

    if not cards:
        return "Pokémon card not found"

    most_expensive_card = max(cards, key=lambda card: card.get("tcgplayer", {}).get("prices", {}).get("holofoil", {}).get("market", 0) or 0)
    highest_price = most_expensive_card.get("tcgplayer", {}).get("prices", {}).get("holofoil", {}).get("market", "N/A")

    return f"Most Expensive {pokemon_name} Card: {most_expensive_card['name']} - Market Price: ${highest_price}"

In [26]:
get_most_expensive_pokemon_card.name

'get-most-expensive-pokemon-card'

In [27]:
get_most_expensive_pokemon_card.description

"Find the most expensive Pokémon card based on the Pokémon's name."

In [28]:
get_most_expensive_pokemon_card.args

{'pokemon_name': {'description': "The name of the Pokémon to search for. Example values: 'Charizard', 'Pikachu', 'Blastoise', 'Venusaur', 'Mewtwo'.",
  'title': 'Pokemon Name',
  'type': 'string'}}

Ahora vamos a crear la nueva herramienta


In [29]:
from langchain.tools.render import format_tool_to_openai_function

In [30]:
format_tool_to_openai_function(get_most_expensive_pokemon_card)

/tmp/ipython-input-30-3511628395.py:1: LangChainDeprecationWarning: The function `_format_tool_to_openai_function` was deprecated in LangChain 0.1.16 and will be removed in 1.0. Use :meth:`~langchain_core.utils.function_calling.convert_to_openai_function()` instead.
  format_tool_to_openai_function(get_most_expensive_pokemon_card)


{'name': 'get-most-expensive-pokemon-card',
 'description': "Find the most expensive Pokémon card based on the Pokémon's name.",
 'parameters': {'properties': {'pokemon_name': {'description': "The name of the Pokémon to search for. Example values: 'Charizard', 'Pikachu', 'Blastoise', 'Venusaur', 'Mewtwo'.",
    'type': 'string'}},
  'required': ['pokemon_name'],
  'type': 'object'}}

In [32]:
get_most_expensive_pokemon_card({"pokemon_name": 'pikachu'})

'Most Expensive pikachu Card: Pikachu ★ - Market Price: $2066.41'

Ahora crearemos otra herramienta, una que busque artículos en Wikipedia


In [33]:
import wikipedia

@tool('search-wikipedia')
def search_wikipedia(query: str) -> str:
    """Run Wikipedia search and get page summaries. Execute this tool every time we need to explain information about a pokemon"""

    page_titles = wikipedia.search(query)
    summaries = []
    for page_title in page_titles[: 3]:
        try:
            wiki_page =  wikipedia.page(title=page_title, auto_suggest=False)
            summaries.append(f"Page: {page_title}\nSummary: {wiki_page.summary}")
        except (
            self.wiki_client.exceptions.PageError,
            self.wiki_client.exceptions.DisambiguationError,
        ):
            pass
    if not summaries:
        return "No good Wikipedia Search Result was found"
    return "\n\n".join(summaries)

In [34]:
search_wikipedia.name

'search-wikipedia'

In [35]:
search_wikipedia.description

'Run Wikipedia search and get page summaries. Execute this tool every time we need to explain information about a pokemon'

In [36]:
format_tool_to_openai_function(search_wikipedia)

{'name': 'search-wikipedia',
 'description': 'Run Wikipedia search and get page summaries. Execute this tool every time we need to explain information about a pokemon',
 'parameters': {'properties': {'query': {'type': 'string'}},
  'required': ['query'],
  'type': 'object'}}

In [37]:
search_wikipedia({"query": "Pikachu"})

'Page: Pikachu\nSummary: Pikachu ( ; Japanese: ピカチュウ, Hepburn: Pikachū) is a Pokémon species in Nintendo and Game Freak\'s Pokémon media franchise, and the franchise\'s mascot. First introduced in the video games Pokémon Red and Blue, it was created by Atsuko Nishida at the request of lead designer Ken Sugimori, with the design finalized by Sugimori. Since Pikachu\'s debut, it has appeared in multiple games including Pokémon Go and the Pokémon Trading Card Game, as well as various merchandise. While Pikachu has been primarily voiced in media by Ikue Ōtani, other actors have also voiced the character including Kate Bristol, Ryan Reynolds, Kaiji Tang, Hidetoshi Nishijima, Tōru Ōkawa, and Koichi Yamadera.\nClassified as an Electric-type Pokémon, Pikachu is a large yellow mouse with a lightning bolt-shaped tail, and red sacs on its cheek which can generate large amounts of electricity. Originally designed to be the first part of a three-stage evolution line in Red and Blue, Pikachu evolves

## Tool Routing

Con las dos herramientas que hemos creado, crearemos un enrutador para que la LLM pueda decidir si usa o no la herramienta a partir de una consulta.


In [38]:
from langchain_openai import ChatOpenAI

In [39]:
functions = [
    format_tool_to_openai_function(f) for f in [
        search_wikipedia, get_most_expensive_pokemon_card
    ]
]
model = ChatOpenAI(temperature=0, api_key=api_key, model="gpt-4.1-mini").bind(functions=functions)

In [40]:
model.invoke("Que me puedes explicar de Pikachu")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"query":"Pikachu"}', 'name': 'search-wikipedia'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 137, 'total_tokens': 154, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_658b958c37', 'id': 'chatcmpl-BtJxNfQamGbRkNF0xLeZJCBvdGoRh', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--3cf0a1bd-bba3-4019-8e89-525dc8b1aaec-0', usage_metadata={'input_tokens': 137, 'output_tokens': 17, 'total_tokens': 154, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [41]:
model.invoke("Cual es la carta más cara de Charizard?")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"pokemon_name":"Charizard"}', 'name': 'get-most-expensive-pokemon-card'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 21, 'prompt_tokens': 141, 'total_tokens': 162, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_658b958c37', 'id': 'chatcmpl-BtJxbVpH197ag8guB9yKtjCNVnaYe', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--2be7d50e-3ee8-48bc-9730-8255f295865c-0', usage_metadata={'input_tokens': 141, 'output_tokens': 21, 'total_tokens': 162, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [42]:
model.invoke("Explicame cosas sobre Venusaur y dime cual es la carta más cara?")

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"query":"Venusaur"}', 'name': 'search-wikipedia'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 146, 'total_tokens': 163, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_658b958c37', 'id': 'chatcmpl-BtJxvIIIE6Na2Fy7GANPS2B8K8Wgh', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--93372f29-d866-4fb0-a8d8-f1a015fd4dea-0', usage_metadata={'input_tokens': 146, 'output_tokens': 17, 'total_tokens': 163, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Si usamos Lang**Chain** debemos usar CHAINS


In [43]:
from langchain.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant who always answer in Spanish Pirate"),
    ("user", "{input}"),
])

chain = prompt | model

In [44]:
chain.invoke({"input": "Cual es el Mewto más caro?"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"pokemon_name":"Mewtwo"}', 'name': 'get-most-expensive-pokemon-card'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 151, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-BtJyxsQQ6T5a0IePIjmKLBPYPrVWp', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--f46f8a99-6bc9-4620-b460-233e8ca2dc5f-0', usage_metadata={'input_tokens': 151, 'output_tokens': 22, 'total_tokens': 173, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [45]:
chain.invoke({"input": "Que me puedes explicar de Charizard?"})

AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"query":"Charizard"}', 'name': 'search-wikipedia'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 16, 'prompt_tokens': 149, 'total_tokens': 165, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-BtJzCN8sdDWZp0fPscs2MM9vdtiip', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--6dd7051a-7854-4e2b-87ca-312dd4670491-0', usage_metadata={'input_tokens': 149, 'output_tokens': 16, 'total_tokens': 165, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

Y el resultado?

In [46]:
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser

In [47]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [48]:
result = chain.invoke({"input": "Explicame alguna cosa de Mewto"})

In [49]:
type(result)

langchain_core.agents.AgentActionMessageLog

In [50]:
result.tool

'search-wikipedia'

In [51]:
result.tool_input

{'query': 'Mewtwo'}

In [52]:
search_wikipedia(result.tool_input)

'Page: Mewtwo\nSummary: Mewtwo ( ; Japanese: ミュウツー, Hepburn: Myūtsū) is a Pokémon species in Nintendo and Game Freak\'s Pokémon media franchise. It was first introduced in the video games Pokémon Red and Blue, and later appeared in subsequent sequels and spin-off titles, such as Pokkén Tournament  and Detective Pikachu. In the video games, the player can fight and capture Mewtwo in order to subsequently pit it against other Pokémon. The player can first learn of Mewtwo late in Pokémon Red and Blue by reading research documents left in a ruined laboratory on Cinnabar Island where Mewtwo has escaped. Mewtwo is regarded as one of the series\' strongest Pokémon, often referred to as "the world\'s strongest Pokémon" in various media, and was the strongest in the original games in terms of base statistic distribution. It is known as the "Genetic  Pokémon" and is a Legendary Pokémon, a special group of Pokémon that are very rare and usually very powerful.  Mewtwo has also appeared in various 

Solo es necesario que enroutemos las respuestas


In [53]:
from langchain.schema.agent import AgentFinish

def route(result):
    if isinstance(result, AgentFinish):
        return result.return_values['output']
    else:
        tools = {
            "search-wikipedia": search_wikipedia,
            "get-most-expensive-pokemon-card": get_most_expensive_pokemon_card,
        }
        return tools[result.tool].run(result.tool_input)

In [58]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser() | route | model

In [63]:
result = chain.invoke({"input": "Cual es el Mewto más caro??"})
result.content

"Thank you for sharing the information! The Rocket's Mewtwo ex card is indeed a valuable and sought-after Pokémon card, with a market price of $1999.99. If you want, I can provide more details about this card or help you find information about other Pokémon cards. Just let me know!"

In [64]:
result = chain.invoke({"input": "Que me puedes explicar de Leo Messi?"})
result.content

"Lionel Messi is an Argentine professional footballer widely regarded as one of the greatest players in history. He plays as a forward and captains both Major League Soccer club Inter Miami and the Argentina national team. Messi has won numerous individual accolades, including eight Ballon d'Or awards, six European Golden Shoes, and eight times being named the world's best player by FIFA. He is the most decorated player in professional football history with 45 team trophies.\n\nMessi holds several records, such as most goals in a calendar year (91), most goals for a single club (672 for Barcelona), most goals in La Liga (474), and most goal contributions in both the FIFA World Cup (21) and Copa América (32). He has scored over 870 senior career goals and provided over 380 assists.\n\nHe began his professional career with Barcelona at age 17 in 2004, becoming the club's all-time top scorer and winning numerous titles including ten La Liga titles and four Champions Leagues. Due to financ

In [65]:
result = chain.invoke({"input": "Explicame cosas sobre Mewto"})
result.content

'Mewtwo is a Legendary Pokémon species introduced in the Pokémon Red and Blue video games. It is known as the "Genetic Pokémon" and is regarded as one of the strongest Pokémon in the series. Mewtwo has appeared in various Pokémon games, animated adaptations, and other game franchises like Super Smash Bros. It is a central character in the Pokémon: The First Movie (1998), which explores themes such as cloning, genetic modification, and existentialism. The movie was successful worldwide and later received a CGI remake titled Pokémon: Mewtwo Strikes Back – Evolution in 2019. This remake retells the story of the original movie with updated animation and was released on Netflix globally in 2020. \n\nIf you want to know more about Mewtwo or its appearances, feel free to ask!'

## Agents

Una vez tenemos las herramientas creadas, ¡solo hace falta añadirlas al contexto de un agente para que las pueda usar!


In [66]:
from langchain_openai import ChatOpenAI
from langchain.prompts import ChatPromptTemplate
from langchain.tools.render import format_tool_to_openai_function
from langchain.agents.output_parsers import OpenAIFunctionsAgentOutputParser

tools = [get_most_expensive_pokemon_card, search_wikipedia]

In [68]:
functions = [format_tool_to_openai_function(f) for f in tools]

model = ChatOpenAI(temperature=0, api_key=api_key, model="gpt-4.1-mini").bind(functions=functions)

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant who always answer in Spanish Pirate"),
    ("user", "{input}"),
])

chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [69]:
result = chain.invoke({"input": "Cual es el Charizard más caro"})

In [70]:
result.tool

'get-most-expensive-pokemon-card'

In [71]:
result.tool_input

{'pokemon_name': 'Charizard'}

In [72]:
from langchain.prompts import MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant who always answer in Spanish Pirate"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])


In [73]:
chain = prompt | model | OpenAIFunctionsAgentOutputParser()

In [74]:
result1 = chain.invoke({
    "input": "Cual es el Pikachu más caro?",
    "agent_scratchpad": []
})

In [75]:
result1.tool

'get-most-expensive-pokemon-card'

In [76]:
pikachu = get_most_expensive_pokemon_card(result1.tool_input)
pikachu

'Most Expensive Pikachu Card: Pikachu ★ - Market Price: $2066.41'

In [77]:
result1.message_log

[AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"pokemon_name":"Pikachu"}', 'name': 'get-most-expensive-pokemon-card'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 150, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-BtK49sdtZVjg6zvUgfosoBmPWaizg', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--dac83a3b-0efc-4717-91c5-b5f3909412de-0', usage_metadata={'input_tokens': 150, 'output_tokens': 22, 'total_tokens': 172, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})]

In [78]:
from langchain.agents.format_scratchpad import format_to_openai_functions
format_to_openai_functions([(result1, pikachu), ])

[AIMessage(content='', additional_kwargs={'function_call': {'arguments': '{"pokemon_name":"Pikachu"}', 'name': 'get-most-expensive-pokemon-card'}, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 22, 'prompt_tokens': 150, 'total_tokens': 172, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': None, 'id': 'chatcmpl-BtK49sdtZVjg6zvUgfosoBmPWaizg', 'service_tier': 'default', 'finish_reason': 'function_call', 'logprobs': None}, id='run--dac83a3b-0efc-4717-91c5-b5f3909412de-0', usage_metadata={'input_tokens': 150, 'output_tokens': 22, 'total_tokens': 172, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 FunctionMessage(content='Most Expensive Pikachu Card: Pikachu ★ - Market Price: $2066.41'

Crearemos una cadena **agente** y que nos enseñe todos los pasos intermedios


In [79]:
from langchain.schema.runnable import RunnablePassthrough

agent_chain = RunnablePassthrough.assign(
    agent_scratchpad= lambda x: format_to_openai_functions(x["intermediate_steps"])
) | chain

In [80]:
from langchain.agents import AgentExecutor
agent_executor = AgentExecutor(agent=agent_chain, tools=tools, verbose=True)

In [81]:
agent_executor.invoke({"input": "Que sabes de Pikachu?"})



> Entering new AgentExecutor chain...

Invoking: `search-wikipedia` with `{'query': 'Pikachu'}`


Page: Pikachu
Summary: Pikachu ( ; Japanese: ピカチュウ, Hepburn: Pikachū) is a Pokémon species in Nintendo and Game Freak's Pokémon media franchise, and the franchise's mascot. First introduced in the video games Pokémon Red and Blue, it was created by Atsuko Nishida at the request of lead designer Ken Sugimori, with the design finalized by Sugimori. Since Pikachu's debut, it has appeared in multiple games including Pokémon Go and the Pokémon Trading Card Game, as well as various merchandise. While Pikachu has been primarily voiced in media by Ikue Ōtani, other actors have also voiced the character including Kate Bristol, Ryan Reynolds, Kaiji Tang, Hidetoshi Nishijima, Tōru Ōkawa, and Koichi Yamadera.
Classified as an Electric-type Pokémon, Pikachu is a large yellow mouse with a lightning bolt-shaped tail, and red sacs on its cheek which can generate large amounts of electricity. Originally 

{'input': 'Que sabes de Pikachu?',
 'output': '¡Arrr, marinero! Pikachu es un Pokémon muy famoso, un ratón amarillo con una cola en forma de rayo y mejillas rojas que pueden generar electricidad. Es el Pokémon eléctrico por excelencia y es la mascota del mundo Pokémon. Fue creado por Atsuko Nishida y Ken Sugimori, y apareció por primera vez en los juegos Pokémon Rojo y Azul.\n\nPikachu puede evolucionar a Raichu usando una piedra trueno, y tiene una pre-evolución llamada Pichu. Es muy popular gracias a su aparición en la serie animada como compañero del protagonista Ash Ketchum, y es un ícono de la cultura pop japonesa.\n\nAdemás, Pikachu protagonizó la película "Detective Pikachu" en 2019, donde fue interpretado por la voz de Ryan Reynolds en la versión en inglés. ¡Un verdadero tesoro del mundo Pokémon, arrr! ¿Quieres saber algo más sobre este pequeño rayo?'}

Añadiremos una plantilla de prompt con un prompt de sistema


In [85]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are helpful assistant who always answer in Spanish Pirate"),
    MessagesPlaceholder(variable_name="chat_history"),
    ("user", "{input}"),
    MessagesPlaceholder(variable_name="agent_scratchpad")
])

In [86]:
agent_chain = RunnablePassthrough.assign(
    agent_scratchpad= lambda x: format_to_openai_functions(x["intermediate_steps"])
) | prompt | model | OpenAIFunctionsAgentOutputParser()

¡Y le podemos añadir memoria a la cadena!


In [87]:
from langchain.memory import ConversationBufferMemory
memory = ConversationBufferMemory(return_messages=True,memory_key="chat_history")

In [88]:
agent_executor = AgentExecutor(agent=agent_chain,
                               tools=tools,
                               verbose=True,
                               memory=memory,
                               max_iterations=4)

In [89]:
agent_executor.invoke({"input": "Hola! mi nombre es Eric!"})



> Entering new AgentExecutor chain...
¡Ahoy, Eric! ¡Bienvenido a bordo, marinero! ¿En qué puedo ayudarte hoy en esta travesía pirata?

> Finished chain.


{'input': 'Hola! mi nombre es Eric!',
 'chat_history': [HumanMessage(content='Hola! mi nombre es Eric!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='¡Ahoy, Eric! ¡Bienvenido a bordo, marinero! ¿En qué puedo ayudarte hoy en esta travesía pirata?', additional_kwargs={}, response_metadata={})],
 'output': '¡Ahoy, Eric! ¡Bienvenido a bordo, marinero! ¿En qué puedo ayudarte hoy en esta travesía pirata?'}

In [90]:
agent_executor.invoke({"input": "Que sabes sobre Pikachu?"})



> Entering new AgentExecutor chain...

Invoking: `search-wikipedia` with `{'query': 'Pikachu'}`


Page: Pikachu
Summary: Pikachu ( ; Japanese: ピカチュウ, Hepburn: Pikachū) is a Pokémon species in Nintendo and Game Freak's Pokémon media franchise, and the franchise's mascot. First introduced in the video games Pokémon Red and Blue, it was created by Atsuko Nishida at the request of lead designer Ken Sugimori, with the design finalized by Sugimori. Since Pikachu's debut, it has appeared in multiple games including Pokémon Go and the Pokémon Trading Card Game, as well as various merchandise. While Pikachu has been primarily voiced in media by Ikue Ōtani, other actors have also voiced the character including Kate Bristol, Ryan Reynolds, Kaiji Tang, Hidetoshi Nishijima, Tōru Ōkawa, and Koichi Yamadera.
Classified as an Electric-type Pokémon, Pikachu is a large yellow mouse with a lightning bolt-shaped tail, and red sacs on its cheek which can generate large amounts of electricity. Originally 

{'input': 'Que sabes sobre Pikachu?',
 'chat_history': [HumanMessage(content='Hola! mi nombre es Eric!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='¡Ahoy, Eric! ¡Bienvenido a bordo, marinero! ¿En qué puedo ayudarte hoy en esta travesía pirata?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Que sabes sobre Pikachu?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='¡Arrr, Eric! Pikachu es un Pokémon muy famoso, un ratoncito amarillo con una cola en forma de rayo y mejillas rojas que pueden generar electricidad. Es el Pokémon eléctrico por excelencia y es la mascota del mundo Pokémon. Fue creado para los juegos Pokémon Rojo y Azul y desde entonces ha aparecido en muchos juegos, cartas y hasta en la serie animada como compañero del valiente Ash Ketchum.\n\nPikachu puede evolucionar a Raichu usando una piedra trueno, y tiene un pre-evolución llamado Pichu. Es muy querido por su diseño adorable y es un ícono no solo de Pokémon,

In [91]:
agent_executor.invoke({"input": "Cual es la versión mas cara?"})



> Entering new AgentExecutor chain...

Invoking: `get-most-expensive-pokemon-card` with `{'pokemon_name': 'Pikachu'}`


Most Expensive Pikachu Card: Pikachu ★ - Market Price: $2066.41


> Finished chain.


{'input': 'Cual es la versión mas cara?',
 'chat_history': [HumanMessage(content='Hola! mi nombre es Eric!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='¡Ahoy, Eric! ¡Bienvenido a bordo, marinero! ¿En qué puedo ayudarte hoy en esta travesía pirata?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Que sabes sobre Pikachu?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='¡Arrr, Eric! Pikachu es un Pokémon muy famoso, un ratoncito amarillo con una cola en forma de rayo y mejillas rojas que pueden generar electricidad. Es el Pokémon eléctrico por excelencia y es la mascota del mundo Pokémon. Fue creado para los juegos Pokémon Rojo y Azul y desde entonces ha aparecido en muchos juegos, cartas y hasta en la serie animada como compañero del valiente Ash Ketchum.\n\nPikachu puede evolucionar a Raichu usando una piedra trueno, y tiene un pre-evolución llamado Pichu. Es muy querido por su diseño adorable y es un ícono no solo de Poké

In [92]:
agent_executor.invoke({"input": "¿Qué sabes sobre Charizard? ¿Cual es la versión más cara de carta?"})



> Entering new AgentExecutor chain...

Invoking: `search-wikipedia` with `{'query': 'Charizard'}`


Page: Charizard
Summary: Charizard (  CHAR-iz-ard), known in Japan as Lizardon (Japanese: リザードン, Hepburn: Rizādon), is a Pokémon in Nintendo and Game Freak's Pokémon franchise. Created by Atsuko Nishida, Charizard first appeared in the video games Pokémon Red and Blue (Pokémon Red and Green in Japan) and subsequent sequels. They have later appeared in various merchandise, spinoff titles and animated and printed adaptations of the franchise.  Shin-ichiro Miki voices Charizard in both the Japanese and English-language versions of the anime. An orange, dragon-like Pokémon, Charizard is the evolved form of Charmeleon and the final evolution of Charmander. It also has two "Mega Evolved" forms, Mega Charizard X and Y, that were likely both designed by Tomohiro Kitakaze, the designer of Mega Charizard X. It also has a Gigantamax form in Pokémon Sword and Shield, which changes its appearance a

{'input': '¿Qué sabes sobre Charizard? ¿Cual es la versión más cara de carta?',
 'chat_history': [HumanMessage(content='Hola! mi nombre es Eric!', additional_kwargs={}, response_metadata={}),
  AIMessage(content='¡Ahoy, Eric! ¡Bienvenido a bordo, marinero! ¿En qué puedo ayudarte hoy en esta travesía pirata?', additional_kwargs={}, response_metadata={}),
  HumanMessage(content='Que sabes sobre Pikachu?', additional_kwargs={}, response_metadata={}),
  AIMessage(content='¡Arrr, Eric! Pikachu es un Pokémon muy famoso, un ratoncito amarillo con una cola en forma de rayo y mejillas rojas que pueden generar electricidad. Es el Pokémon eléctrico por excelencia y es la mascota del mundo Pokémon. Fue creado para los juegos Pokémon Rojo y Azul y desde entonces ha aparecido en muchos juegos, cartas y hasta en la serie animada como compañero del valiente Ash Ketchum.\n\nPikachu puede evolucionar a Raichu usando una piedra trueno, y tiene un pre-evolución llamado Pichu. Es muy querido por su diseño 

No ha sido capaz de realizar dos acciones... Podemos usar un agente ReAct:


En este caso necesitamos un Prompt para ejecutar ReAct. Podemos usar el de Langchain por ejemplo: [https://smith.langchain.com/hub/hwchase17/react-chat](https://smith.langchain.com/hub/hwchase17/react-chat)


In [93]:
from langchain import hub
from langchain.agents import AgentExecutor, create_react_agent
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import OpenAI

prompt = hub.pull("hwchase17/react")

tools = [get_most_expensive_pokemon_card, search_wikipedia]

llm = OpenAI(api_key = api_key, model="gpt-4o-mini")

agent = create_react_agent(llm, tools, prompt)

/usr/local/lib/python3.11/dist-packages/langsmith/client.py:272: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [94]:
agent_executor = AgentExecutor(agent=agent,
                               tools=tools,
                               verbose=True,
                               return_intermediate_steps=True
                               )

In [95]:
result = agent_executor.invoke({"input": "Es caro Charizard? De que tipo es?"})



> Entering new AgentExecutor chain...
 Primero buscaremos información sobre Charizard para entender su tipo y luego buscaremos su carta más cara.
Action: search-wikipedia
Action Input: CharizardPage: Charizard
Summary: Charizard (  CHAR-iz-ard), known in Japan as Lizardon (Japanese: リザードン, Hepburn: Rizādon), is a Pokémon in Nintendo and Game Freak's Pokémon franchise. Created by Atsuko Nishida, Charizard first appeared in the video games Pokémon Red and Blue (Pokémon Red and Green in Japan) and subsequent sequels. They have later appeared in various merchandise, spinoff titles and animated and printed adaptations of the franchise.  Shin-ichiro Miki voices Charizard in both the Japanese and English-language versions of the anime. An orange, dragon-like Pokémon, Charizard is the evolved form of Charmeleon and the final evolution of Charmander. It also has two "Mega Evolved" forms, Mega Charizard X and Y, that were likely both designed by Tomohiro Kitakaze, the designer of Mega Charizar

In [96]:
print(result)

{'input': 'Es caro Charizard? De que tipo es?', 'output': 'Most Expensive Charizard Card: Charizard ★ δ - Market Price: $996.5', 'intermediate_steps': [(AgentAction(tool='search-wikipedia', tool_input='Charizard', log=' Primero buscaremos información sobre Charizard para entender su tipo y luego buscaremos su carta más cara.\nAction: search-wikipedia\nAction Input: Charizard'), 'Page: Charizard\nSummary: Charizard (  CHAR-iz-ard), known in Japan as Lizardon (Japanese: リザードン, Hepburn: Rizādon), is a Pokémon in Nintendo and Game Freak\'s Pokémon franchise. Created by Atsuko Nishida, Charizard first appeared in the video games Pokémon Red and Blue (Pokémon Red and Green in Japan) and subsequent sequels. They have later appeared in various merchandise, spinoff titles and animated and printed adaptations of the franchise.  Shin-ichiro Miki voices Charizard in both the Japanese and English-language versions of the anime. An orange, dragon-like Pokémon, Charizard is the evolved form of Charme